#Preliminary Analysis of Habitat Modification Scores 
####(main dataset is rss_tbl_river_habitat_survey - hms_class and subscores)

##River Habitat Survey HMS class

In [0]:
# Load packages
library(sparklyr) # connect to databricks
library(data.table) # data management
# GAMS - use mgcv for ordinal/catagorial data
#library(mgcv)
# Create a sparklyr connection.
sc <- spark_connect(method = "databricks")

##Load data
set parameters

In [0]:
# set parameters
param <- list(st="RSN")

# River habitat survey
# hms_class
rss_dt <- sdf_sql(sc, paste0("SELECT *
FROM prd_dash_lab.seda_restricted.rss_tbl_river_habitat_survey
WHERE network_id LIKE '", param$st, 
"%' AND completeness_band NOT LIKE '%Unacceptable%' ")) |> as.data.table()

In [0]:
# change to factors
rss_dt[, network_id := as.factor(network_id)]


In [0]:
#return only hms_class and hms_class_desc columns
rss_dt[, .(hms_class, hms_class_desc), keyby=hms_class] |> unique()

In [0]:
hms_class_table <- table(rss_dt[, hms_class])
# as %
hms_class_table  <- hms_class_table  / sum(hms_class_table) * 100
# Add names
classes <- c("Pristine/Near-natural", "Predominantly unmodified", "Obviously modified", "Significantly modified", "Severely modified")
names(hms_class_table) <- classes
# View table
hms_class_table

In [0]:
library(ggplot2)
hms_class_table
# Convert the table into a data frame
hms_plot_data <- data.frame(hms_class = names(hms_class_table),percentage = as.numeric(hms_class_table))

cols <- c("#2ca25f", "#99d8c9", "#fec44f", "#fe9929", "#d95f0e")
#plot horizontal bar plot 
ggplot(hms_plot_data, aes(x = percentage, y = "",fill = hms_class)) + geom_col(colour = "black", width = 0.6) +
  scale_fill_manual(values = cols) +
  scale_x_continuous(breaks = seq(0, 100, 20),limits = c(0, 100),expand = expansion(mult = c(0, 0.05))) + 
  labs(x = "Proportion of Rivers (%)", y =  "HMS Class", fill = "HMS Class") +
  #geom_text(aes(label = paste0(round(percentage, 1), "%")),vjust = -0.4,size = 3.5) +
  theme_bw() +
  theme(
    axis.text.x = element_text(
      angle = 45,
      hjust = 1,
      size = 10),
    legend.position = "bottom")


In [0]:
# Create bar plot
ggplot(hms_plot_data, aes(x = hms_class, y = percentage,fill = hms_class)) + geom_col(colour = "black") +
  scale_fill_manual(values = cols) +
  scale_y_continuous(breaks = seq(0, 100, 20),limits = c(0, 100),expand = expansion(mult = c(0, 0.05))) + 
  labs(x = "HMS Class", y = "Proportion of Rivers (%)",fill = "HMS Class") +
  geom_text(aes(label = paste0(round(percentage, 1), "%")),vjust = -0.4,size = 3.5) +
  theme_bw() +
  theme(axis.text.x = element_text(
      angle = 45,
      hjust = 1,
      size = 10),
    legend.position = "none")

In [0]:
# Calculate overall % meeting high quality or above
hms_class_table[1] + hms_class_table[2]

###Calculate confidence intervals for the classes
*on the raw data

In [0]:
# Uses the binomial test to generate confidence intervals from the raw class data
# Thi gives class's proportion with a 95% CI
res <- rss_dt[, .N, by = hms_class][order(hms_class)]
res[, p := N / sum(N)]
res[, c("lo", "hi") := {
  ci <- sapply(N, \(n) binom.test(n, sum(N))$conf.int)
  .(ci[1, ], ci[2, ])
}]

# Remove row with na values for now.
res <- res[-6,]
res

In [0]:
#plot with confidence intervals
hms_plot_data$lower_ci <- res$lo * 100
hms_plot_data$upper_ci <- res$hi * 100

ggplot(hms_plot_data, aes(x = hms_class, y = percentage,fill = hms_class)) + geom_col(colour = "black") +
  scale_fill_manual(values = cols) +
  scale_y_continuous(breaks = seq(0, 100, 20),limits = c(0, 100),expand = expansion(mult = c(0, 0.05))) + 
  labs(x = "HMS Class", y = "Proportion of Rivers (%)",fill = "HMS Class") +
  geom_errorbar(aes(ymin = lower_ci,ymax = upper_ci),width = 0.1,linewidth = 0.25) +  # Add 95% confidence intervals
  geom_text(aes(label = paste0(round(percentage, 1), "%")),vjust = -1.5,size = 3.5) +
  theme_bw() +
  theme(axis.text.x = element_text(
      angle = 45,
      hjust = 1,
      size = 10),
    legend.position = "none")

####Visualise score range

In [0]:

# Relationship between hqa_score and hqa_class
hms_dt <- rss_dt[,.(hms_score, hms_class, hms_class_desc)]

par(mar = c(5, 12, 4, 2))   # bottom, left, top, right
#x:hms score, y:hms class or hms desc
plot(hms_dt[, .(hms_score, hms_class)], bg=adjustcolor("blue", 0.5), pch=21, , cex=2, axes=FALSE)
axis(1)
axis(2, at=1:5, labels=c("Pristine", "Predominantly unmodified", "Obviously modified", "Significantly modified", "Severely modified"), las = 1,cex.axis = 0.8)
#mtext("HMS class", side = 2, line = 8)  # adjust the x-axis label

For HMS scores range from 0 to >6000, larger the score more modifed the river reach. 
0–16: Pristine
17–199: Predominantly unmodified
200–499: Obviously modified
500–1399: Significantly modified
≥1400: Severly modified

See Appendix one from river habitat survey website : https://www.riverhabitatsurvey.org/RHSfiles/RHSToolboxHelp/RiverHabitatSurveyToolbox.html?Indices.html

In [0]:
#is a boxplot better to visualise the spread of points
#also note that are no adjested scores provided for HMS - unlikde HQA
ggplot(rss_dt, aes(factor(hms_class), hms_score, group = hms_class)) +
  geom_boxplot(aes(fill = factor(hms_class)), alpha = 0.7, outlier.shape = NA) +
  geom_jitter(aes(colour = factor(hms_class)), width = 0.15, alpha = 0.5, size = 1.5) +
  scale_fill_manual(values = cols) +
  scale_colour_manual(values = cols) +
  labs(x = "HMS Class", y = "HMS Score") +
  theme_bw() +
  theme(legend.position = "none")


In [0]:
rss_dt[duplicated(hms_score) | duplicated(hms_score, fromLast = TRUE),
       .(network_id, hms_score, hms_class, completeness_band)][order(hms_score)]

rss_dt[is.na(hms_class), .N] #34 observations have no HMS_class or HMS_score but have complteness score >7, these were removed before the PCA
rss_dt[is.na(hms_score), .N]

In [0]:
#count number of networks in each hms_class
hms_network_counts <- rss_dt[
  !is.na(hms_class),
  .(n_networks = uniqueN(network_id)),
  by = .(hms_class, hms_class_desc)
][order(hms_class)]

hms_network_counts #34 is NA

###Duplicates in the data, should'nt be really since the survey was only done once. 5 networks have duplicates

In [0]:

duplicate_networks <- rss_dt[
  !is.na(network_id), .N,by = network_id][N > 1][order(-N)]

duplicate_networks

rss_dt[
  duplicated(network_id) | duplicated(network_id, fromLast = TRUE)
][order(network_id)]

####Plot features by class

In [0]:
# Melt data to long format to create boxplots (all)
#subset the columns that you want
data.table(number = seq_along(names(rss_dt)),column = names(rss_dt))
rss_dt_sub <- rss_dt[, c(2, 9:19,63 ), with = FALSE]
#remove cells 1 (network id) and 13 (hms desc)
long <- melt(rss_dt_sub[,-c(1, 13)], id.vars = "hms_class",
             variable.name = "metric", value.name = "value")

# remove lines with NA in HMS class
long <- long[!is.na(hms_class)]
unique(long$hms_class)

In [0]:
# Make sure HMS class is an ordered factor
# Labels and colours
hc <- c("Pristine", "Predominantly unmodified", "Obviously modified", "Significantly modified", "Severely modified")
#hms_cols <- c("#2ca25f", "#99d8c9", "#fec44f", "#fe9929", "#d95f0e")

long[, hms_class := factor(hms_class,labels = hc, levels = 1:5)]

ggplot(long, 
aes(x = hms_class, y = value, fill = hms_class)) +
  geom_boxplot(outlier.colour = "grey50",outlier.size = 0.5) +
  facet_wrap(~ metric, scales = "free_y", ncol = 4) +
  scale_fill_manual(values = cols) +
  labs(x = "HMS Class",y = "Value",fill = "HMS Class") +
  theme_bw() +
  theme( legend.position = "none",
    axis.text.x = element_text(
      angle = 45,
      hjust = 1,
      size = 8),
    strip.text = element_text(size = 9))

In [0]:
# Just very high and very low classes
long2 <- long[hms_class %in% c("Pristine", "Severely modified")]
long2[, hms_class := droplevels(hms_class)]

ggplot(long2, 
aes(x = hms_class, y = value, fill = hms_class)) +
  geom_boxplot(outlier.colour = "grey50",outlier.size = 0.5) +
  facet_wrap(~ metric, scales = "free_y", ncol = 4) +
  scale_fill_manual(values = cols[c(1,5)]) +
  labs(x = "HMS Class",y = "Value",fill = "HMS Class") +
  theme_bw() +
  theme( legend.position = "none",
    axis.text.x = element_text(
      angle = 45,
      hjust = 1,
      size = 8),
    strip.text = element_text(size = 9))


In [0]:
# Wilcox test - is there a significant difference between the classes (best + worst)?
sapply(metrics, function(m) {
  d <- long2[metric == m]
  wilcox.test(value ~ hms_class, data = d, exact = FALSE)$p.value |> p.adjust("BY")})

In [0]:
# Calculate raw p-values
metrics <- levels(long$metric)
p_raw <- sapply(metrics, function(m) {d <- long2[metric == m]
    wilcox.test(
    value ~ hms_class,
    data = d,
    exact = FALSE
  )$p.value})

# Apply BY correction across all tests
p_by <- p.adjust(p_raw, method = "BY")

# Results table
wilcox_results <- data.table(
  metric = names(p_raw),
  p_value = as.numeric(p_raw),
  p_adjusted_BY = as.numeric(p_by),
  significant = p_by < 0.05)

wilcox_results[order(p_adjusted_BY)]

##PCA - Identifying Habitat Subclasses Driving Modification Class

In [0]:
#list subclasses here as a vector
names(rss_dt)
hms_vars <- c("hms_reforced_bnk_bed_subscore", "hms_rsctned_bnk_subscore", "hms_berms_embnkments_subscore", "hms_culverts_subscore", "hms_wds_subscore", "hms_bridges_subscore", "hms_poaching_subscore", "hms_fords_subscore", "hms_ofall_dflector_subscore")

colSums(is.na(rss_dt[, ..hms_vars]))

#list the neworks that have missing data
rss_dt[is.na(hms_ofall_dflector_subscore),
       .(network_id, hms_class, survey_date)]

#remove NAs, create new data table
rss_pca <- rss_dt[complete.cases(rss_dt[, ..hms_vars])]

#PCA omitting the NAs
pca <- prcomp(rss_pca[, ..hms_vars], scale. = TRUE, center=TRUE)


scores <- as.data.table(pca$x)
scores[, hms_class := rss_pca$hms_class]

summary(pca)

##Plot PCA

Scree plot - Variance explained by each principal component

In [0]:
#plot scree plot - shows how much variance each PC explains
plot(pca, type = "l")

Scatter plot - Principal component scores coloured by habitat modification class

In [0]:
install.packages("ggrepel")
library(ggrepel)

# Scores and HMS classes
scores <- as.data.table(pca$x)
scores[, HMS := factor(rss_pca$hms_class, levels = 1:5, labels = classes)]

# Convex hull for each class
hulls <- scores[, .SD[chull(PC1, PC2)], by = HMS]

# Loadings and arrow scale
loads <- data.table(
  variable = rownames(pca$rotation),
  pca$rotation[, 1:2])
loads[, `:=`(PC1 = PC1 * 8, PC2 = PC2 * 8)]

ggplot(scores, aes(PC1, PC2, colour = HMS)) +
  geom_polygon(data = hulls, aes(fill = HMS, group = HMS),alpha = 0.2,colour = NA) +
  geom_segment(data = loads,aes(x = 0, y = 0, xend = PC1, yend = PC2),inherit.aes = FALSE,arrow = arrow(length = unit(0.15, "cm")),colour = "black") +
  geom_text_repel(data = loads,aes(PC1, PC2, label = variable),inherit.aes = FALSE,size = 3) +
  geom_point(alpha = 0.7) +
  scale_colour_manual(values = cols) +
  scale_fill_manual(values = cols) +
  labs(x = "PC1 (19.0%)", y = "PC2 (16.0%)",
    colour = "HMS Class",
    fill = "HMS Class") +
  theme_bw()

Boxplot of PC1 by HMS Class - Distribution of PC1 scores across habitat modification classes

In [0]:
par(mar = c(10, 4, 4, 2))

boxplot(
  PC1 ~ HMS,
  data = scores,
  las = 2)

  scores

Contribution of HMS subclass scores to PCA axes

hms_reforced_bnk_bed_subscore, hms_wds_subscore and hms_bridges_subscore primarily associated with PC1 

hms_rsctned_bnk_subscore, hms_berms_embnkments_subscore and hms_poaching_subscore associated with PC2

In [0]:
pca$rotation

In [0]:
sort(abs(pca$rotation[,1]), decreasing = TRUE)


In [0]:
sort(abs(pca$rotation[,2]), decreasing = TRUE)

###Correlation test of the principal component using  Spearman, apply Benjamini-Yekutieli as correction 

In [0]:
pcs <- c("PC1", "PC2", "PC3", "PC4")

tests <- lapply(pcs, function(pc) {

  # Keep complete observations for this PC
  d <- scores[
    !is.na(HMS) & !is.na(get(pc))]
    cor.test(as.numeric(d$HMS), d[[pc]],method = "spearman",exact = FALSE)})

# Extract results
rho <- sapply(tests, function(x) unname(x$estimate))
p_values <- sapply(tests, function(x) x$p.value)

# Create results table and apply BY correction
results <- data.table(PC = pcs, rho = rho,
  p_value = p_values,
  p_adjusted_BY = p.adjust(p_values, method = "BY"),
  significant = p.adjust(p_values, method = "BY") < 0.05)

results

In [0]:
names(scores)

Spearman show a strong positive relationship association is strong (tau and rho) and highly significant (p < 0.001)

####Correlation for subclass variables 

In [0]:

tests <- lapply(hms_vars, function(v) {
    d <- rss_pca[
    !is.na(hms_class) & !is.na(get(v))]
  cor.test(
    x = as.numeric(d$hms_class),
    y = d[[v]],
    method = "spearman",
    exact = FALSE)})

rho <- sapply(tests, function(x) unname(x$estimate))
p_raw <- sapply(tests, function(x) x$p.value)
p_by <- p.adjust(p_raw, method = "BY")

cor_results <- data.table(
  metric = hms_vars,
  n = sapply(hms_vars, function(v) {
    rss_pca[!is.na(hms_class) & !is.na(get(v)), .N]}),
  rho = as.numeric(rho),
  p_value = as.numeric(p_raw),
  p_adjusted_BY = as.numeric(p_by),
  significant = p_by < 0.05)

cor_results

Only hms_ford_subscore is not significant. hms_rsctned_bnk_subscore has the highest rho - strongest association (relates to the PCA we see above)

In [0]:
### Correlation matrix function
# Default method = spearman
# p adjustment method = BY
corr_matrix <- function(x, method = "spearman", adjust = "BY", min_pairs = 3) {
  df   <- as.data.frame(x)
  vars <- names(df)
  n    <- length(vars)
  r <- p <- cnt <- matrix(NA_real_, n, n, dimnames = list(vars, vars))

  for (i in seq_len(n)) {
    for (j in i:n) {
      ok <- is.finite(df[[i]]) & is.finite(df[[j]])
      cnt[i, j] <- cnt[j, i] <- sum(ok)
      if (i == j) { r[i, j] <- 1; next }
      if (sum(ok) < min_pairs) next
      ct <- suppressWarnings(
        cor.test(df[[i]][ok], df[[j]][ok], method = method, exact = FALSE)
      )
      r[i, j] <- r[j, i] <- unname(ct$estimate)
      p[i, j] <- p[j, i] <- ct$p.value
    }
  }

  p_adj <- p
  ut <- upper.tri(p)
  p_adj[ut] <- p.adjust(p[ut], method = adjust)
  p_adj[lower.tri(p_adj)] <- t(p_adj)[lower.tri(p_adj)]

  list(r = r, p = p, p_adj = p_adj, n = cnt, method = method, adjust = adjust)
}

### Plot function
plot_corr_matrix <- function(res, digits = 2, palette = "Blue-Red 2",
                             use_adj = TRUE, cex = 0.5, mar = c(12, 12, 3, 2),
                             main = NULL) {
  r    <- res$r
  p    <- if (use_adj) res$p_adj else res$p
  vars <- rownames(r)
  n    <- length(vars)

  stars <- ifelse(is.na(p), "",
           ifelse(p < 0.001, "***",
           ifelse(p < 0.01,  "**",
           ifelse(p < 0.05,  "*", ""))))
  lab <- matrix(paste0(formatC(r, format = "f", digits = digits), stars), n, n)
  lab[is.na(r)] <- ""

  rp <- r;   diag(rp) <- NA
  diag(lab) <- ""

  cols   <- hcl.colors(100, palette, rev = TRUE)   # -1 = red, +1 = blue
  breaks <- seq(-1, 1, length.out = 101)

  op <- par(mar = mar)
  on.exit(par(op))
  image(1:n, 1:n, t(rp[n:1, ]), col = cols, breaks = breaks,
        axes = FALSE, xlab = "", ylab = "", main = main)
  axis(1, 1:n, vars,      las = 2, tick = FALSE)
  axis(2, 1:n, rev(vars), las = 2, tick = FALSE)
  box()

  g <- expand.grid(y = 1:n, x = 1:n)
  text(g$x, g$y, as.vector(lab[n:1, ]), cex = cex)
  invisible(res)
}

In [0]:
# Run correlation on data
res_hms <- corr_matrix(rss_dt_sub[,-c(1,13)], method = "spearman")


In [0]:
# Plot results
plot_corr_matrix(res_hms, main = "Spearman correlations (BY-adjusted)")
# ignore hqa_Score and adjusted for now
abline(h=10:11, col="red", lwd=2)
abline(v=1, col="red", lwd=2)
# Draw a box around the imporant hqa class correlations
rect(1.5, 0.5, 2.5, 9.5, lwd=3, xpd=NA)

Numbers are rho values
*refer to p-values
Blue: positive correlation. As one variable increases, the other tends to increase.
Red/pink: negative correlation. As one variable increases, the other tends to decrease.
White or pale grey: correlation close to zero.
Darker colour: stronger correlation, closer to -1 or +1.
Lighter colour: weaker correlation, closer to 0.

####Look at how features interact

In [0]:
# Analysis function which combines features
# regression
combo_corr <- function(x, target, predictors = setdiff(names(x), target),
                       max_k = 3, method = "spearman") {
  df <- as.data.frame(x)
  df <- df[complete.cases(df[c(target, predictors)]), c(target, predictors)]
  if (method == "spearman") df[] <- lapply(df, rank)   # rank-based, Spearman-style
  df[predictors] <- scale(df[predictors])              # so sums aren't scale-dominated

  res <- do.call(rbind, lapply(2:max_k, function(k) {
    do.call(rbind, lapply(combn(predictors, k, simplify = FALSE), function(v) {
      s <- summary(lm(reformulate(v, target), data = df))
      f <- s$fstatistic
      data.frame(combo   = paste(v, collapse = " + "),
                 k       = k,
                 r_sum   = cor(rowSums(df[v]), df[[target]]),
                 R2      = s$r.squared,
                 adj_R2  = s$adj.r.squared,
                 p_model = pf(f[1], f[2], f[3], lower.tail = FALSE),
                 row.names = NULL)
    }))
  }))
  res$p_adj <- p.adjust(res$p_model, method = "BY")
  res[order(-res$adj_R2), ]
}

In [0]:
# Choose which features and number of combinations - try 3 for now
# Dont use features which are "the same" (highly correlated)
res_hms_com <- combo_corr(rss_dt_sub[,-c(1, 13)], target = "hms_class",
    predictors = c("hms_reforced_bnk_bed_subscore", "hms_rsctned_bnk_subscore", "hms_berms_embnkments_subscore", "hms_culverts_subscore", "hms_wds_subscore", "hms_bridges_subscore", "hms_poaching_subscore", "hms_fords_subscore", "hms_ofall_dflector_subscore"),
    max_k = 3)
# Results are ordered by r2 value 
head(res_hms_com, 10)

In [0]:
# Trying to understand the interactions...
# Ordinal logistic regression - this looks at interactions between pairs
library(MASS)

ord_interaction_scan <- function(x, target, predictors, min_n = 30) {
  df <- as.data.frame(x)[c(target, predictors)]
  df <- df[complete.cases(df), ]
  df[[target]] <- factor(df[[target]], levels = 1:5)
  df[predictors] <- lapply(df[predictors], function(z) as.numeric(scale(z)))

  out <- do.call(rbind, lapply(combn(predictors, 2, simplify = FALSE), function(v) {
    m0 <- polr(as.formula(sprintf("`%s` ~ `%s` + `%s`", target, v[1], v[2])),
               data = df, Hess = TRUE)
    m1 <- polr(as.formula(sprintf("`%s` ~ `%s` * `%s`", target, v[1], v[2])),
               data = df, Hess = TRUE)
    lrt <- anova(m0, m1)
    data.frame(a = v[1], b = v[2],
               beta_int = tail(coef(m1), 1),     # log-odds change per SD of the other
               LR       = lrt$`LR stat.`[2],
               p_int    = lrt$`Pr(Chi)`[2],
               AIC_add  = AIC(m0), AIC_full = AIC(m1),
               row.names = NULL)
  }))
  out$p_adj <- p.adjust(out$p_int, method = "BY")
  out[order(out$p_int), ]}

In [0]:
res_ord <- ord_interaction_scan(rss_dt_sub[,-c(1, 13)], "hms_class", 
c("hms_reforced_bnk_bed_subscore", "hms_rsctned_bnk_subscore", "hms_berms_embnkments_subscore", "hms_culverts_subscore", "hms_wds_subscore", "hms_bridges_subscore", "hms_poaching_subscore", "hms_fords_subscore", "hms_ofall_dflector_subscore"))

Repeated warnings that fitted probabilities are numerically 0 or 1, indicating complete or near-complete separation. Diagnostic check showed that 31 of 36 interaction pairs were excluded because of warnings or failure. 

#All code below is exploratory - IGNORE!!
###More Variables added
####more environmental factors extracted from rss_tbl_samp_invertebrate 
##### !!re-do this bit and consider mutliple points for the same network id, maybe choose the values that are closest in date to when the HMS was recorded!!

In [0]:
#load data from prd_dash_lab.seda_restricted.rss_tbl_samp_invertebrate
#rss_invert_dt <- sdf_sql(sc, paste0("SELECT *
#FROM prd_dash_lab.seda_restricted.rss_tbl_samp_invertebrate
#WHERE network_id LIKE '", param$st, 
#"%' AND completeness_band NOT LIKE '%Unacceptable%' ")) |> as.data.table()

rss_invert_dt <- sdf_sql(sc, paste0("SELECT *
FROM prd_dash_lab.seda_restricted.rss_tbl_samp_invertebrate
WHERE network_id LIKE '", param$st, 
"%'")) |> as.data.table()

In [0]:
#choose variables from invert table to use in PCA
names(rss_invert_dt)
#primary_bank_use, primary_land_use -- too many catergories/too complex try again
#turbidity, shade, flow, detritus, overlay_silt_cover
invert_vars <- rss_invert_dt[, .(network_id, turbidity, shade, flow, detritus, overlay_silt_cover)]

for (v in names(invert_vars)) {
  cat("\n", v, "\n")
  print(sort(unique(invert_vars[[v]])))}


In [0]:
#convert them to ordinal in their natural order
invert_vars[, turbidity := factor(
  turbidity,
  levels = c("CLEAR: Water not visibly turbid", "SLIGHT: Visible turbidity but no significant effect on light penetration", "MODERATE: Significant effect on light penetration", "HIGH: Visibility limited to 10cm depth"),
  ordered = TRUE)]

invert_vars[, shade  := factor(
  shade,
  levels = c("NONE: Site is unshaded", "LIGHT: < 25%", "MODERATE: 25 - 50%", "HEAVY: > 50%"),
  ordered = TRUE)]  

invert_vars[, flow := factor(
  flow,
  levels = c("NO FLOW - Still water/unconnected pools", "DRY: No water", "LOW: Flow present but below normal level", "NORMAL: Normal flow rate", "HIGH: Faster than normal rates", "SPATE: Flood or near flood conditions, water course near bankfull"),
  ordered = TRUE)]

invert_vars[, detritus := factor(
  detritus,
  levels = c("NONE: None present", "LOCAL: Occasional patches - < 30% of area", "WIDESPREAD: 30 - 60% of area", "EXTENSIVE: > 61% of area."),
  ordered = TRUE)]

#invert_vars[, overlay_silt_cover := factor(
#  overlay_silt_cover,
#  levels = c("NONE: None present", "LOCAL: Occasional patches - < 30% of area", "WIDESPREAD: 30 - 60% of area", "EXTENSIVE: > 61% of area."),
#  ordered = TRUE)]


In [0]:
levels(invert_vars$turbidity)
levels(invert_vars$shade)
levels(invert_vars$flow)
levels(invert_vars$detritus)

In [0]:
hms_lookup <- unique(
  rss_dt[, .(network_id, hms_class, hms_class_desc)])

hms_lookup[, .N, by = network_id][N > 1]

In [0]:
There are some repeated networks - remove them

In [0]:
dup_ids <- rss_dt[, .N, by = network_id][N > 1, network_id]
#dup_ids
# RSN1807 RSN1129 RSN1132 RSN1262 RSN1946

hms_lookup <- rss_dt[
  !network_id %in% dup_ids,
  .(network_id, hms_class, hms_class_desc)]

hms_lookup[, .N, by = network_id][N > 1]

invert_vars <- merge(
  invert_vars,
  hms_lookup,
  by = "network_id",
  all.x = TRUE)
  names(invert_vars)

head(invert_vars)

In [0]:
env_vars <- c(
  "turbidity","shade","flow","detritus")

#covert to numeric
invert_num <- copy(invert_vars)

invert_num[, (env_vars) := lapply(.SD, as.numeric),
           .SDcols = env_vars]
#check
str(invert_num[, ..env_vars])

#check for NAs
colSums(is.na(invert_num[, ..env_vars]))

#remove NAs
pca_invert_data <- invert_num[complete.cases(invert_num[, ..env_vars])]

#PCA omitting the NAs
pca_invert <- prcomp(pca_invert_data[, ..env_vars],scale. = TRUE)

summary(pca_invert)

###Plot PCA

In [0]:
plot(pca_invert, type = "l")

In [0]:
scores_invert <- as.data.table(pca_invert$x)
scores_invert[, hms_class := pca_invert_data$hms_class]

#plot PCA with arrows(subclasses)

scores[, hms_class_desc := factor(
  hms_class,
  levels = 1:5,
  labels = c(
   classes))]

cols <- c("darkgreen", "green", "yellow", "orange", "red")

#addiing arrows to pca plot
plot(scores_invert$PC1, scores_invert$PC2,
col = cols[scores_invert$hms_class],pch = 16,xlab = "PC1 (31.9%)", ylab = "PC2 (28.5%)")

# Add arrows
arrows(0, 0, 
pca_invert$rotation[,1] * 5,
pca_invert$rotation[,2] * 5,
length = 0.1,col = "red")

# Add labels
text(
  pca_invert$rotation[,1] * 5,
  pca_invert$rotation[,2] * 5,
  labels = rownames(pca_invert$rotation),
  col = "black",
  pos = 4,
  cex = 0.8)

legend(
  "topleft",
  inset = c(0, 0),
  legend = c(
    "Pristine/Near-natural",
    "Predominantly unmodified",
    "Obviously modified",
    "Significantly modified",
    "Severely modified"),
  col = cols, pch = 16, bty = "n", cex=0.8, xpd=TRUE)


Does not seem like there is a strong associationi between the PCA and HMS class

In [0]:
sort(abs(pca_invert$rotation[,1]), decreasing = TRUE)


In [0]:
sort(abs(pca_invert$rotation[,2]), decreasing = TRUE)

In [0]:
cor.test(
  as.numeric(scores_invert$hms_class),
  scores_invert$PC1,
  method = "kendall")

Although it is significant tau<-0.1 is negligible, and no meaningful relationship :(  PC2 resulted in similar values p<0.001 tau<-0.1

Spearman or Kendall test?

In [0]:
cor.test(
  as.numeric(pca_invert_data$hms_class),
  pca_invert_data$turbidity,
  method = "spearman")

cor.test(
  as.numeric(pca_invert_data$hms_class),
  pca_invert_data$shade,
  method = "spearman")

cor.test(
  as.numeric(pca_invert_data$hms_class),
  pca_invert_data$flow,
  method = "spearman")

cor.test(
  as.numeric(pca_invert_data$hms_class),
  pca_invert_data$detritus,
  method = "spearman")

The correlations above resulted in significant p values but rho <0.1 - therefore negligible/weak relationship


Trying primary_land_use and primary_bank_use, unsure if this has worked. Some observations have 2 or more values, Eg simple... and complex...is the second one supposed to be secondary bank use?

In [0]:
use_vars <- rss_invert_dt[, .(network_id, primary_land_use, primary_bank_use)]

In [0]:
use_vars_1 <- merge(
  use_vars,
  hms_lookup,
  by = "network_id",
  all.x = TRUE)
  names(use_vars_1)

In [0]:
chisq.test(
  table(
    use_vars_1$primary_bank_use,
    use_vars_1$hms_class))

In [0]:
tab <- prop.table(
  table(use_vars_1$primary_bank_use,
        use_vars_1$hms_class), margin = 1)

round(tab, 2)